In [1]:
# 0. IMPORTS
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.ensemble import IsolationForest
from sklearn.metrics import (
    precision_score, recall_score, roc_auc_score, 
    average_precision_score, confusion_matrix, classification_report
)
import joblib

In [2]:
# 1. LOAD THE EXACT SAME TRAIN/TEST SPLIT FROM NOTEBOOK 04
X_train = pd.read_parquet('/kaggle/input/datasets/hamaz911/notebook-5-dataset/X_train.parquet')
X_test = pd.read_parquet('/kaggle/input/datasets/hamaz911/notebook-5-dataset/X_test.parquet')
y_train = pd.read_parquet('/kaggle/input/datasets/hamaz911/notebook-5-dataset/y_train.parquet')['isFraud']
y_test = pd.read_parquet('/kaggle/input/datasets/hamaz911/notebook-5-dataset/y_test.parquet')['isFraud']

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train fraud rate:", y_train.mean() * 100)
print("y_test fraud rate:", y_test.mean() * 100)

X_train shape: (472432, 311)
X_test shape: (118108, 311)
y_train fraud rate: 3.5135215226741625
y_test fraud rate: 3.4409184813899145


In [3]:
# QUICK CHECK: CONFIRM COLUMN COUNT AND CONTENTS LOOK RIGHT
print(X_train.columns.tolist())
print("\nAny unexpected columns like 'isFraud' or 'TransactionID' still present?")
print([c for c in X_train.columns if c in ['isFraud', 'TransactionID', 'TransactionDT', 'DeviceInfo']])

['TransactionAmt', 'ProductCD', 'card1', 'card2', 'card3', 'card4', 'card5', 'card6', 'addr1', 'addr2', 'dist1', 'dist2', 'P_emaildomain', 'R_emaildomain', 'C1', 'C2', 'C3', 'C4', 'C5', 'C6', 'C7', 'C8', 'C9', 'C10', 'C11', 'C12', 'C13', 'C14', 'D1', 'D2', 'D3', 'D4', 'D5', 'D6', 'D7', 'D8', 'D9', 'D10', 'D11', 'D12', 'D13', 'D14', 'D15', 'M1', 'M2', 'M3', 'M4', 'M5', 'M6', 'M7', 'M8', 'M9', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10', 'V12', 'V13', 'V14', 'V15', 'V17', 'V19', 'V20', 'V23', 'V24', 'V25', 'V26', 'V27', 'V29', 'V31', 'V35', 'V36', 'V37', 'V38', 'V39', 'V40', 'V41', 'V42', 'V44', 'V45', 'V46', 'V47', 'V48', 'V51', 'V53', 'V54', 'V55', 'V56', 'V59', 'V61', 'V62', 'V64', 'V65', 'V66', 'V67', 'V68', 'V69', 'V73', 'V75', 'V76', 'V77', 'V78', 'V79', 'V80', 'V82', 'V83', 'V85', 'V86', 'V87', 'V88', 'V90', 'V95', 'V96', 'V98', 'V99', 'V100', 'V104', 'V105', 'V107', 'V108', 'V109', 'V110', 'V111', 'V112', 'V113', 'V114', 'V115', 'V116', 'V117', 'V118', 'V119', 'V1

In [4]:
# 2. APPROACH A: TRY DIFFERENT scale_pos_weight VALUES
# Default 'balanced' approx equals: (negative count / positive count)
imbalance_ratio = (y_train == 0).sum() / (y_train == 1).sum()
print("Natural imbalance ratio:", imbalance_ratio)

Natural imbalance ratio: 27.46147358274595


In [5]:
# 3. TEST MULTIPLE scale_pos_weight VALUES
weight_options = {
    'natural_ratio': imbalance_ratio,      # ~27.5 (same as 'balanced')
    'half_ratio': imbalance_ratio / 2,      # ~13.7 (less aggressive toward catching fraud)
    'double_ratio': imbalance_ratio * 2     # ~55 (more aggressive toward catching fraud)
}

results_weighting = {}

for name, weight in weight_options.items():
    print(f"\n--- Training with scale_pos_weight = {weight:.2f} ({name}) ---")
    model = lgb.LGBMClassifier(
        n_estimators=100,
        scale_pos_weight=weight,
        random_state=42,
        verbose=-1  # suppress repetitive training logs since we're running this 3 times
    )
    model.fit(X_train, y_train)
    
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]
    
    results_weighting[name] = {
        'precision': precision_score(y_test, y_pred),
        'recall': recall_score(y_test, y_pred),
        'roc_auc': roc_auc_score(y_test, y_proba),
        'pr_auc': average_precision_score(y_test, y_proba)
    }
    print(results_weighting[name])


--- Training with scale_pos_weight = 27.46 (natural_ratio) ---
{'precision': 0.20001315010848839, 'recall': 0.7485236220472441, 'roc_auc': np.float64(0.9027480977520902), 'pr_auc': np.float64(0.5011510455710066)}

--- Training with scale_pos_weight = 13.73 (half_ratio) ---
{'precision': 0.313800792864222, 'recall': 0.6232775590551181, 'roc_auc': np.float64(0.9031800219755629), 'pr_auc': np.float64(0.5039990389329009)}

--- Training with scale_pos_weight = 54.92 (double_ratio) ---
{'precision': 0.1318906157780838, 'recall': 0.8248031496062992, 'roc_auc': np.float64(0.8946406572390764), 'pr_auc': np.float64(0.48695436656018204)}


In [6]:
# 4a. INSTALL/IMPORT SMOTE
from imblearn.over_sampling import SMOTE

# 4b. PREP NUMERIC-SAFE VERSION FOR SMOTE (reuse same approach as Logistic Regression prep)
X_train_smote_prep = X_train.copy()
cat_cols_all = X_train_smote_prep.select_dtypes(include=['object', 'category']).columns.tolist()

for col in cat_cols_all:
    X_train_smote_prep[col] = X_train_smote_prep[col].astype('category').cat.codes

X_train_smote_prep = X_train_smote_prep.fillna(-999)

print("Any NaN remaining?", X_train_smote_prep.isnull().sum().sum())
print("Any non-numeric remaining?", X_train_smote_prep.select_dtypes(exclude=[np.number]).columns.tolist())

Any NaN remaining? 0
Any non-numeric remaining? []


In [7]:
# 4c. APPLY SMOTE TO TRAINING DATA ONLY
smote = SMOTE(random_state=42, sampling_strategy=0.3)  # bring minority class up to 30% of majority, not full 50/50

X_train_smote, y_train_smote = smote.fit_resample(X_train_smote_prep, y_train)

print("Before SMOTE:", y_train.value_counts())
print("\nAfter SMOTE:", y_train_smote.value_counts())

Before SMOTE: isFraud
0    455833
1     16599
Name: count, dtype: int64

After SMOTE: isFraud
0    455833
1    136749
Name: count, dtype: int64


In [8]:
# 4d. TRAIN LIGHTGBM ON SMOTE-REBALANCED DATA
lgb_smote_model = lgb.LGBMClassifier(
    n_estimators=100,
    random_state=42,
    verbose=-1
)
lgb_smote_model.fit(X_train_smote, y_train_smote)

print("Model trained.")

Model trained.


In [9]:
# 4e. EVALUATE — MUST USE THE SAME NUMERIC-ENCODED PREP FOR X_TEST (since SMOTE model trained on that format)
X_test_smote_prep = X_test.copy()
for col in cat_cols_all:
    X_test_smote_prep[col] = X_test_smote_prep[col].astype('category').cat.codes
X_test_smote_prep = X_test_smote_prep.fillna(-999)

y_pred_smote = lgb_smote_model.predict(X_test_smote_prep)
y_proba_smote = lgb_smote_model.predict_proba(X_test_smote_prep)[:, 1]

print("Precision:", precision_score(y_test, y_pred_smote))
print("Recall:", recall_score(y_test, y_pred_smote))
print("ROC-AUC:", roc_auc_score(y_test, y_proba_smote))
print("PR-AUC:", average_precision_score(y_test, y_proba_smote))

Precision: 0.7231668437832094
Recall: 0.3348917322834646
ROC-AUC: 0.8815843750181239
PR-AUC: 0.4619689383112361


In [10]:
# 5a. PREP DATA FOR ISOLATION FOREST (also needs numeric, non-null input, like SMOTE)
# We can reuse the same numeric-encoded prep from Step 4

# 5b. TRAIN ISOLATION FOREST (UNSUPERVISED — doesn't use y_train at all during fitting)
iso_forest = IsolationForest(
    n_estimators=100,
    contamination=0.035,  # our known ~3.5% fraud rate, tells the model roughly what % to flag as anomalies
    random_state=42,
    n_jobs=-1
)

iso_forest.fit(X_train_smote_prep)  # reusing our numeric-safe prep from Step 4, fits on ALL data (not just legit)

print("Model trained.")

Model trained.


In [11]:
# 5c. ALTERNATIVE: TRAIN ISOLATION FOREST ONLY ON LEGITIMATE TRANSACTIONS
X_train_legit_only = X_train_smote_prep[y_train == 0]

iso_forest_legit_only = IsolationForest(
    n_estimators=100,
    contamination=0.035,
    random_state=42,
    n_jobs=-1
)
iso_forest_legit_only.fit(X_train_legit_only)

print("Model trained on legit-only data.")

Model trained on legit-only data.


In [12]:
# 5d. EVALUATE BOTH ISOLATION FOREST VERSIONS
# Isolation Forest's predict() returns -1 for anomaly, 1 for normal — need to convert to match our 0/1 fraud labels
def evaluate_iso_forest(model, X_test_data, y_test_data, label):
    raw_pred = model.predict(X_test_data)
    y_pred_iso = np.where(raw_pred == -1, 1, 0)  # -1 (anomaly) -> 1 (fraud), 1 (normal) -> 0 (not fraud)
    
    # decision_function gives anomaly score; more negative = more anomalous
    # we flip sign so higher score = more likely fraud, consistent with our other models
    y_score_iso = -model.decision_function(X_test_data)
    
    print(f"--- {label} ---")
    print("Precision:", precision_score(y_test_data, y_pred_iso))
    print("Recall:", recall_score(y_test_data, y_pred_iso))
    print("ROC-AUC:", roc_auc_score(y_test_data, y_score_iso))
    print("PR-AUC:", average_precision_score(y_test_data, y_score_iso))
    print()

evaluate_iso_forest(iso_forest, X_test_smote_prep, y_test, "Trained on MIXED data (incl. fraud)")
evaluate_iso_forest(iso_forest_legit_only, X_test_smote_prep, y_test, "Trained on LEGIT-ONLY data")

--- Trained on MIXED data (incl. fraud) ---
Precision: 0.08999786734911495
Recall: 0.10383858267716535
ROC-AUC: 0.7528329554372163
PR-AUC: 0.0917031571005084

--- Trained on LEGIT-ONLY data ---
Precision: 0.10776846915460776
Recall: 0.1392716535433071
ROC-AUC: 0.754067763630117
PR-AUC: 0.09429919461561348



In [13]:
# 6. SAVE NOTEBOOK 05 RESULTS AND THE CHOSEN MODEL
import json

notebook05_results = {
    'lightgbm_natural_ratio': {'precision': 0.2000, 'recall': 0.7485, 'roc_auc': 0.9027, 'pr_auc': 0.5012},
    'lightgbm_half_ratio': {'precision': 0.3138, 'recall': 0.6233, 'roc_auc': 0.9032, 'pr_auc': 0.5040},
    'lightgbm_double_ratio': {'precision': 0.1319, 'recall': 0.8248, 'roc_auc': 0.8946, 'pr_auc': 0.4870},
    'lightgbm_smote': {'precision': 0.7232, 'recall': 0.3349, 'roc_auc': 0.8816, 'pr_auc': 0.4620},
    'isolation_forest_mixed': {'precision': 0.0900, 'recall': 0.1038, 'roc_auc': 0.7528, 'pr_auc': 0.0917},
    'isolation_forest_legit_only': {'precision': 0.1078, 'recall': 0.1393, 'roc_auc': 0.7541, 'pr_auc': 0.0943}
}

with open('/kaggle/working/notebook05_results.json', 'w') as f:
    json.dump(notebook05_results, f, indent=2)

# Save the chosen model (natural ratio, our carry-forward choice) for Notebook 06
final_lgb_model = lgb.LGBMClassifier(
    n_estimators=100,
    scale_pos_weight=imbalance_ratio,  # ~27.46
    random_state=42,
    verbose=-1
)
final_lgb_model.fit(X_train, y_train)  # native categorical/NaN handling, no SMOTE

joblib.dump(final_lgb_model, '/kaggle/working/lgb_notebook05_model.pkl')
print("Saved results and model.")

Saved results and model.
